# Player Number-Guided AI Highlight Video Curator
**CSCI 5922 — Mateusz Muszynski & Colin Wallace**

Tracks jersey **#6 (dark kit)** through *Denver vs. Virginia | 2025 ACC Men's Soccer*
and assembles a personal highlight reel using real SoccerNet training data.

### Setup checklist (first run only)
1. **Settings → Accelerator → GPU T4 x1**
2. **Settings → Internet → On**
3. Add your match video as dataset `soccer-match-video`
4. Add-ons → Secrets → add `SOCCERNET_EMAIL` and `SOCCERNET_PASS`
5. Run cells top-to-bottom

In [ ]:
# ============================================================
# CONFIGURATION — edit this cell, then Run All
# ============================================================

TARGET_JERSEY = 19
JERSEY_COLOR  = None    # set 'dark'/'light' if needed

VIDEO_PATH    = '/kaggle/input/soccer-match-video/denver.mp4'

# Set True  → delete any existing weights and retrain from scratch with real SoccerNet data
# Set False → skip training if valid weights already exist in /kaggle/working/models/
FORCE_RETRAIN = True

# Video trim (skips pre-game intro)
TRIM_START_SEC    = 300   # 5 min
TRIM_DURATION_SEC = 1200  # 20 min

# Pipeline settings
FRAME_SKIP           = 2     # process every 2nd frame (halves memory pressure)
CONF_THRESHOLD       = 0.25
HIGHLIGHT_THRESHOLD  = 0.40

# ============================================================
import os, pathlib
OUTPUT_DIR  = '/kaggle/working/outputs'
OUTPUT_PATH = f'{OUTPUT_DIR}/highlights_jersey{TARGET_JERSEY}.mp4'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Target: jersey #{TARGET_JERSEY} ({JERSEY_COLOR} kit)')
print(f'FORCE_RETRAIN = {FORCE_RETRAIN}')


---
## 1 — GPU check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠  No GPU — Settings → Accelerator → GPU T4, then re-run all')


---
## 2 — Clone repo & install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/mateusz-muszynski/csci5922-highlight-curator.git'
REPO_DIR = '/kaggle/working/csci5922'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull --rebase')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')
os.system('git log --oneline -3')


In [ ]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'ultralytics>=8.2.0',
    'supervision>=0.21.0',
    'opencv-python-headless',
    'albumentations',
    'ffmpeg-python',
    'imageio[ffmpeg]',
    'PyYAML', 'tqdm', 'rich',
], check=True)

import torch, torchvision, ultralytics, cv2
print(f'torch {torch.__version__} | tv {torchvision.__version__} | '
      f'ultralytics {ultralytics.__version__} | cv2 {cv2.__version__}')


---
## 3 — Download SoccerNet jersey data

Requires `SOCCERNET_EMAIL` and `SOCCERNET_PASS` Kaggle Secrets.
Skip this cell if data is already present (the cell checks automatically).

In [ ]:
import os, pathlib
os.chdir(REPO_DIR)

# ── Load credentials from Kaggle Secrets ─────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _sc             = UserSecretsClient()
    SOCCERNET_EMAIL = _sc.get_secret("SOCCERNET_EMAIL")
    SOCCERNET_PASS  = _sc.get_secret("SOCCERNET_PASS")
    print(f"Secrets loaded: {SOCCERNET_EMAIL[:4]}****")
except Exception as e:
    raise RuntimeError(
        f"Could not load Kaggle Secrets ({e}).\n"
        "Go to Add-ons → Secrets and add SOCCERNET_EMAIL + SOCCERNET_PASS."
    )

# ── Count existing images ─────────────────────────────────────────────────────
JERSEY_DIR = pathlib.Path(REPO_DIR) / "data" / "soccernet_jersey"

def _count_images(d):
    if not d.exists():
        return 0
    return sum(
        len(list(p.glob("*.jpg")) + list(p.glob("*.png")))
        for p in d.iterdir() if p.is_dir()
    )

img_count = _count_images(JERSEY_DIR)

if img_count >= 1000:
    print(f"Jersey dataset already present ({img_count:,} images) — skipping download.")
else:
    print(f"Downloading SoccerNet jersey-2023 (train split) ... ({img_count} images so far)")
    os.system(
        f'python scripts/download_soccernet.py '
        f'--task jersey --data-dir data/ '
        f'--username "{SOCCERNET_EMAIL}" --password "{SOCCERNET_PASS}"'
    )

img_count   = _count_images(JERSEY_DIR)
class_count = len([p for p in JERSEY_DIR.iterdir() if p.is_dir()]) if JERSEY_DIR.exists() else 0
print(f"\nJersey dataset: {class_count} classes · {img_count:,} images")

if img_count < 500:
    print("WARNING: Very few images downloaded. Check credentials and re-run this cell.")


---
## 4 — Train models

**Jersey CNN** trains on real SoccerNet jersey crops (50 epochs, ~60–90 min on T4).  
**Scorer LSTM** trains on synthetic action clips (~5 min).

Set `FORCE_RETRAIN = False` in the Config cell to skip this after the first run.

In [ ]:
import os, shutil, torch, pathlib, gc
os.chdir(REPO_DIR)

MODELS_DIR      = os.path.join(REPO_DIR, "models")
WEIGHTS_OUT_DIR = "/kaggle/working/models"
os.makedirs(MODELS_DIR,      exist_ok=True)
os.makedirs(WEIGHTS_OUT_DIR, exist_ok=True)

jersey_weights = os.path.join(MODELS_DIR, "jersey_cnn.pt")
scorer_weights = os.path.join(MODELS_DIR, "scorer_lstm.pt")

def _weights_valid(path, prefix="backbone."):
    if not os.path.exists(path):
        return False
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
        keys  = list(state.keys())
        return len(keys) > 0 and any(k.startswith(prefix) for k in keys)
    except Exception as exc:
        print(f"  [WARN] {path}: {exc}")
        return False

def _real_data_available():
    d = pathlib.Path(REPO_DIR) / "data" / "soccernet_jersey"
    if not d.exists():
        return False
    total = sum(
        len(list(p.glob("*.jpg")) + list(p.glob("*.png")))
        for p in d.iterdir() if p.is_dir()
    )
    return total >= 500

# ── FORCE_RETRAIN: wipe existing weights so training always runs ──────────────
if FORCE_RETRAIN:
    for fname in ("jersey_cnn.pt", "scorer_lstm.pt"):
        for search_dir in (MODELS_DIR, WEIGHTS_OUT_DIR):
            p = os.path.join(search_dir, fname)
            if os.path.exists(p):
                os.remove(p)
                print(f"Deleted stale weights: {p}")

jersey_ok = _weights_valid(jersey_weights)
scorer_ok = _weights_valid(scorer_weights)

if not jersey_ok or not scorer_ok:
    if _real_data_available():
        img_count = sum(
            len(list(p.glob("*.jpg")) + list(p.glob("*.png")))
            for p in (pathlib.Path(REPO_DIR) / "data" / "soccernet_jersey").iterdir()
            if p.is_dir()
        )
        print(f"\nReal SoccerNet data found ({img_count:,} images) → training with --kaggle-full")
        print("  Jersey CNN : 50 epochs on real crops  (~60–90 min on T4)")
        print("  Scorer LSTM: 200 synthetic clips,  15 epochs (~5 min on T4)\n")

        if not jersey_ok:
            os.system("python run_training.py --kaggle-full --stages jersey --device cuda --fail-fast")
        if not scorer_ok:
            os.system("python run_training.py --kaggle      --stages scorer  --device cuda --fail-fast")
    else:
        print("\nNo real SoccerNet data — run Cell 3 first to download jersey images.")
        print("Falling back to synthetic stubs as a smoke-test...")
        os.system("python run_training.py --kaggle --stages jersey scorer --device cuda --fail-fast")
else:
    print("Weights already valid — skipping training.  (Set FORCE_RETRAIN=True to retrain.)")

# ── Verify ────────────────────────────────────────────────────────────────────
assert _weights_valid(jersey_weights), "jersey_cnn.pt missing or has wrong keys — training failed"
assert _weights_valid(scorer_weights), "scorer_lstm.pt missing or has wrong keys — training failed"

# ── Copy to /kaggle/working/models for Output tab ────────────────────────────
for fname in ("jersey_cnn.pt", "scorer_lstm.pt"):
    shutil.copy(os.path.join(MODELS_DIR, fname), os.path.join(WEIGHTS_OUT_DIR, fname))

j_state = torch.load(jersey_weights, map_location="cpu", weights_only=True)
s_state = torch.load(scorer_weights, map_location="cpu", weights_only=True)
print(f"\njerscy_cnn.pt  : {os.path.getsize(jersey_weights)/1e6:.1f} MB  first key: {list(j_state.keys())[0]!r}")
print(f"scorer_lstm.pt : {os.path.getsize(scorer_weights)/1e6:.1f} MB  first key: {list(s_state.keys())[0]!r}")
print(f"\n✓ Weights saved to Output tab → {WEIGHTS_OUT_DIR}/")
print("  After this run: Output tab → Save version → save /kaggle/working/models/ as a dataset.")
print("  Next session:   set FORCE_RETRAIN = False and attach that dataset to skip retraining.")


---
## 5 — Verify video & trim gameplay clip

In [ ]:
import os
os.chdir(REPO_DIR)

assert os.path.exists(VIDEO_PATH), (
    f"Video not found: {VIDEO_PATH}\n"
    "Add a Kaggle dataset named 'soccer-match-video' with the .mp4 file."
)
print(f"Video : {VIDEO_PATH}")
print(f"Size  : {os.path.getsize(VIDEO_PATH)/1e9:.2f} GB")

CLIP_PATH = f'/kaggle/working/game_clip_jersey{TARGET_JERSEY}.mp4'
if not os.path.exists(CLIP_PATH):
    print(f"Trimming {TRIM_DURATION_SEC}s clip starting at {TRIM_START_SEC}s ...")
    os.system(
        f'ffmpeg -y -ss {TRIM_START_SEC} -i "{VIDEO_PATH}" -t {TRIM_DURATION_SEC} '
        f'-c copy "{CLIP_PATH}" -loglevel error'
    )
print(f"Clip  : {CLIP_PATH}  ({os.path.getsize(CLIP_PATH)/1e6:.1f} MB)")
VIDEO_CLIP = CLIP_PATH


---
## 6 — Run highlight pipeline

In [ ]:
import os, sys, gc, torch
os.chdir(REPO_DIR)

# Reload src modules fresh (avoids stale state if re-running)
for mod in list(sys.modules.keys()):
    if mod.startswith('src') or mod == 'main':
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from main import run_pipeline

result = run_pipeline(
    video_path               = VIDEO_CLIP,
    jersey_number            = TARGET_JERSEY,
    output_path              = OUTPUT_PATH,
    config_path              = os.path.join(REPO_DIR, 'config.yaml'),
    device                   = 'cuda',
    conf_override            = CONF_THRESHOLD,
    highlight_thresh_override= HIGHLIGHT_THRESHOLD,
    frame_skip               = FRAME_SKIP,
    clip_length_override     = 90,
    clip_stride_override     = 15,
    jersey_color             = JERSEY_COLOR,
    debug_video              = None,
)
print(f'\nHighlight reel → {result}')


---
## 7 — Preview highlight reel

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

if result and os.path.exists(result):
    PREVIEW = f'/kaggle/working/reel_preview_jersey{TARGET_JERSEY}.mp4'
    os.system(
        f'ffmpeg -y -i "{result}" -vcodec libx264 -acodec aac '
        f'-crf 26 -preset fast "{PREVIEW}" -loglevel error'
    )
    data = open(PREVIEW, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(f'''
    <h3>Highlight reel — jersey #{TARGET_JERSEY}</h3>
    <video width="960" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))
else:
    print(f'No highlights found for jersey #{TARGET_JERSEY}.')
    print('Try lowering HIGHLIGHT_THRESHOLD to 0.30 in the Config cell and re-running Cell 6.')


---
## 8 — Jersey frequency scan (optional diagnostic)
Samples 300 frames to verify #6 is being detected with the dark-kit filter.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from collections import Counter
import os, sys
os.chdir(REPO_DIR)

from src.detector import PlayerDetector
from src.jersey_reader import JerseyReader

det = PlayerDetector(conf_threshold=CONF_THRESHOLD, device='cuda')
jr  = JerseyReader(
    model_path=os.path.join(REPO_DIR, 'models/jersey_cnn.pt'),
    color_hint=JERSEY_COLOR,
    device='cuda',
)

cap     = cv2.VideoCapture(VIDEO_CLIP)
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
counter = Counter()
indices = np.linspace(0, total - 1, min(300, max(1, total // 10)), dtype=int)

for fi in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
    ret, frame = cap.read()
    if not ret:
        continue
    for d in det.detect(frame):
        crop = det.crop_upper_body(frame, d)
        num, conf = jr.read(crop)
        if num is not None:
            counter[num] += 1
cap.release()

top20 = counter.most_common(20)
print("Top 20 jersey sightings:")
for num, cnt in top20:
    marker = " ← TARGET" if num == TARGET_JERSEY else ""
    print(f"  #{num:2d}: {cnt}{marker}")

print(f"\n#{TARGET_JERSEY} sightings: {counter.get(TARGET_JERSEY, 0)} / {len(indices)} sampled frames")

nums, cnts = zip(*top20) if top20 else ([], [])
colors = ['green' if n == TARGET_JERSEY else 'steelblue' for n in nums]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([str(n) for n in nums], cnts, color=colors)
ax.set_xlabel('Jersey number')
ax.set_ylabel('Sightings (dark-kit filter active)')
ax.set_title(f'Jersey frequency scan — #{TARGET_JERSEY} highlighted in green')
plt.tight_layout()
plt.savefig(f'/kaggle/working/jersey_freq_{TARGET_JERSEY}.png', dpi=150)
plt.show()
